# AI Tutor — **qwen3-4b multi-turn** · original 30 scenarios (`v1`) · two-call mode

Measures the **2026-08-03 qwen3-4b bottleneck fixes** on the pre-expansion
multi-turn benchmark. Single model — the tag the offline (Jetson) tutor
actually ships:

| tag | note |
| --- | --- |
| `qwen3-4b-jetson` | the settled offline tutor (qwen3:4b-instruct + pinned num_ctx) |

- **Scenarios:** `--multi-turn --subset v1` — the ORIGINAL 30 multi-turn scenarios (tag `v1`),
  every model sees all 30. No sampling.
- **Call mode:** `TUTOR_CALL_MODE=two` — same as the mt50 board, so score
  deltas are attributable to the fixes, not the call-mode change.
- **Tutor** = the local qwen under test; **student-sim + rubric judge** =
  Anthropic (needs `ANTHROPIC_API_KEY`).
- **Branch `offline-optimization`** must be pushed and must carry the fixes — Cell 2
  prints HEAD; check it.

**Before you start**
1. Runtime → **Change runtime type** → **T4 GPU** (the model is ~2.5 GB).
2. Colab Secrets (🔑 sidebar), *Notebook access ON*:
   - `GH_TOKEN` — GitHub classic PAT, `repo` scope (collaborator on `eai6/ai-tutor`).
   - `ANTHROPIC_API_KEY` — **required** (student-sim + rubric judge).
   - `GOOGLE_API_KEY`, `OPENAI_API_KEY` — grader/judge fallback cascade.

## Cell 1 — GPU + mount Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone the repo (branch `offline-optimization` — carries the fixes)

In [ ]:
from google.colab import userdata
import subprocess, os
tok = (userdata.get('GH_TOKEN') or '').strip()
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'offline-optimization', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())
print('HEAD:', subprocess.run(['git','log','-1','--oneline'],capture_output=True,text=True).stdout.strip())

## Cell 3 — fix hardcoded laptop paths

In [ ]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps + start Ollama
Version-pinned: tool-call parsing is Ollama-version-sensitive (memory: ollama-sweep-gotchas).

In [ ]:
!pip install -q -r requirements.txt
# zstd is REQUIRED: Ollama's Linux artifact is now .tar.zst — install.sh pipes
# through zstd and the legacy .tgz URL 404s for pinned versions (verified
# 2026-08-03: ollama-linux-amd64.tgz?version=0.30.7 -> 404, .tar.zst -> 200).
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
import subprocess, time, shutil, os
OLLAMA_VERSION = '0.30.7'
def _has_ollama(): return shutil.which('ollama') is not None
def _install_ollama():
    if _has_ollama(): return True
    for i in range(1, 4):
        print(f'[ollama] pinned install attempt {i}', flush=True)
        subprocess.run(f'curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh', shell=True)
        if _has_ollama(): return True
        time.sleep(5)
    for i in range(1, 4):
        print(f'[ollama] direct-binary attempt {i}', flush=True)
        r = subprocess.run(f'curl -fL --retry 5 --retry-all-errors --connect-timeout 30 '
                           f'-o /tmp/ollama.tar.zst "https://ollama.com/download/ollama-linux-amd64.tar.zst?version={OLLAMA_VERSION}"',
                           shell=True)
        if r.returncode != 0:
            print(f'[ollama] curl exited {r.returncode}', flush=True)
        if os.path.exists('/tmp/ollama.tar.zst') and os.path.getsize('/tmp/ollama.tar.zst') > 1_000_000:
            subprocess.run('tar --zstd -C /usr -xf /tmp/ollama.tar.zst', shell=True)
            if _has_ollama(): return True
        time.sleep(5)
    return False
assert _install_ollama(), "ollama install failed — Runtime -> Disconnect and delete runtime, then retry."
print(subprocess.run(['ollama','--version'],capture_output=True,text=True).stdout.strip())
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash','-c','ollama list'], capture_output=True).returncode == 0:
        print('ollama ready'); break
    time.sleep(2)
else:
    print('ollama NOT ready — check /content/ollama.log')

## Cell 5 — write `.env` from Colab Secrets

In [ ]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n")
print('.env written')

## Cell 6 — fresh DB + eval fixtures

In [ ]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

## Cell 7 — persist results to Drive (symlink → survives disconnects)
Symlinks `offline_eval/multi_turn_results/qwen_mt30/` to `ai-tutor-eval-multiturn/qwen_mt30/` on Drive. Resume-safe: run_matrix.sh skips any model that already has a JSON.

In [ ]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-multiturn/qwen_mt30
!rm -rf offline_eval/multi_turn_results/qwen_mt30 && mkdir -p offline_eval/multi_turn_results && ln -s /content/drive/MyDrive/ai-tutor-eval-multiturn/qwen_mt30 offline_eval/multi_turn_results/qwen_mt30
import os, glob
print('this run writes to:', os.path.realpath('offline_eval/multi_turn_results/qwen_mt30'))
done = sorted(os.path.basename(p)[:-5] for p in glob.glob('offline_eval/multi_turn_results/qwen_mt30/*.json'))
print('already scored:', done or '(none yet)')

## Cell 8 — write the model list (local qwen, Modelfile-pinned tags)
run_matrix.sh detects `infra/ollama/Modelfile.<tag>` and builds the tag via `ollama create` from its registry base instead of pulling it.

In [ ]:
open('offline_eval/models.txt', 'w').write('''# Qwen mt30 — Modelfile-pinned local tags
qwen3-4b-jetson      jetson
''')
print(open('offline_eval/models.txt').read())

## Cell 9 — run the eval (30 v1 multi-turn scenarios, qwen3-4b-jetson)
`TUTOR_CALL_MODE=two` pins two-call (mt50-comparable). Builds the tag from its Modelfile, runs 30 sessions, saves JSON+log to Drive.

In [ ]:
!TUTOR_CALL_MODE=two RESULTS_DIR=$PWD/offline_eval/multi_turn_results/qwen_mt30 SIMPLE_TUTOR_ENGINE=1 CLEANUP_MODELS=1 \
  MODE="--multi-turn --subset v1" bash offline_eval/run_matrix.sh

## Cell 10 — results: pass rate + session end-reasons + mean rubric

In [ ]:
import json, glob, os
from collections import Counter
def rub(r):
    items = (r.get('rubric_result') or {}).get('items') or []
    app = [i for i in items if i.get('applicable')]
    return sum(i['score'] for i in app)/len(app) if app else None
rows = []
for f in sorted(glob.glob('offline_eval/multi_turn_results/qwen_mt30/*.json')):
    d = json.load(open(f)); res = d.get('results') or []
    n = len(res); k = sum(bool(r.get('passed')) for r in res)
    reasons = Counter((r.get('sim_reason') or '?') for r in res)
    scores = [s for s in (rub(r) for r in res) if s is not None]
    mean_rub = sum(scores)/len(scores) if scores else 0.0
    rows.append((os.path.basename(f)[:-5], k, n, mean_rub, dict(reasons)))
print(f"{'MODEL':<22}{'PASS':>8}  {'RUBRIC':>7}   SESSION END-REASONS")
print('-'*76)
for m, k, n, mr, reasons in sorted(rows, key=lambda r: -(r[1]/r[2] if r[2] else 0)):
    print(f"{m:<22}{k:>4}/{n:<3}  {mr:>6.3f}   {reasons}")
if not rows:
    print("(no results yet — run Cell 9)")

## Cell 11 — **did the 2026-08-03 fixes fire?** (grep the logs)
Each line is a fix leaving a trace. Expectations vs the mt50 baseline:
- `percent/decimal scale equivalence` — grader crediting decimal-form answers (mt50 had 5 false negatives).
- `prepended missing … ack` — silent-pose turns getting an acknowledgement (62 in mt50).
- `ack_rotate` — 'Exactly —' chains broken.
- `reveal_filter` — should be MORE frequent than mt50's 3 (now covers no-verdict turns + option paraphrases).
- `repeat_pose rejected … posed 2x` — the per-stem cap.
- `premature_pose blocked … graded incorrect this turn` — same-turn hint guard.
- `auto_pivot` — stuck slots pivoted (attempts OR age).

In [ ]:
import glob, os, re
PATTERNS = {
  'grader scale-equiv': r'percent/decimal scale equivalence',
  'ack prepended': r'prepended missing \w+ ack',
  'ack rotated': r'ack_rotate: repeated opener',
  'reveal redacted': r'reveal_filter: redacted',
  'stem cap': r'already posed \d+x this session',
  'same-turn hint guard': r'graded incorrect this turn',
  'auto pivot': r'auto_pivot: replaced stuck slot',
  'call2 repair': r'call2_repair: Call 1 skipped',
}
logs = sorted(glob.glob('offline_eval/multi_turn_results/qwen_mt30/*.log'))
if not logs:
    print("(no logs yet — run Cell 9)")
else:
    hdr = f"{'MODEL':<22}" + ''.join(f"{k[:15]:>17}" for k in PATTERNS)
    print(hdr); print('-'*len(hdr))
    for lg in logs:
        txt = open(lg, errors='ignore').read()
        counts = [len(re.findall(p, txt)) for p in PATTERNS.values()]
        print(f"{os.path.basename(lg)[:-4]:<22}" + ''.join(f"{c:>17}" for c in counts))

## Reading this run

**Baseline:** mt50 scored `qwen3:4b` 44/50 (88%) on a different (50-scenario,
seed-5) draw — the closest apples-to-apples reference for THIS scenario set is
a fresh run, so treat this board as the new baseline for the original-30.

**What "the fixes worked" looks like:** completions stay at/near 30/30, mean
rubric rises (mt50's tail was 0.42–0.76 driven by false corrections, silent
poses, reveals, and re-asks), and Cell 11 shows the nets firing in sensible
volumes — a handful each, not storms. A net firing hundreds of times means the
model is fighting the harness; investigate before trusting the score.

At n=30 the binomial SE at p≈0.9 is ~5.5pp — a pass-rate change under ~11pp
vs a prior run of this board is noise; lean on mean rubric + end-reasons.